# m9 resolution check

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

In [ ]:
import os, sys, time
from pathlib import Path
PROJECT  = Path(os.environ.get('LG_PROJECT', r'<research-project>'))
sys.path.insert(0, str(PROJECT / 'analogues'))

In [ ]:
import numpy as np, pickle, hdfstream
import flamingo_lg_pipeline as flp

t0 = time.time()
root = hdfstream.open("cosma", "/")
print(f"connected in {time.time()-t0:.1f}s")
print("top level:", list(root))
print("\nFLAMINGO runs available:")
for name in sorted(list(root["FLAMINGO"])):
    print("  ", name)

In [ ]:
print("inside FLAMINGO/L1_m9:")
for n in sorted(list(root["FLAMINGO/L1_m9"])):
    print("  ", n)

print("\ninside FLAMINGO/L1_m8 (for comparison):")
for n in sorted(list(root["FLAMINGO/L1_m8"])):
    print("  ", n)

In [ ]:
print("\nEAGLE:", sorted(list(root["EAGLE"])))

In [ ]:
import numpy as np, pickle, hdfstream
import flamingo_lg_pipeline as flp

t0 = time.time()
root = hdfstream.open("cosma", "/")
print(f"connected in {time.time()-t0:.1f}s")
print("m8 reference path:", flp.DEFAULT_SOAP_PATH)

In [ ]:
for run in ["L1_m9", "Jet", "fgas-4sigma", "Planck"]:
    p = f"FLAMINGO/L1_m9/{run}"
    try:
        inner = sorted(list(root[p]))
        print(f"{run:<14} -> {inner}")
        if "SOAP-HBT" in inner:
            snaps = sorted(list(root[f"{p}/SOAP-HBT"]))
            print(f"{'':<14}    SOAP-HBT: {snaps[:3]} ... {snaps[-3:]}  ({len(snaps)} files)")
    except Exception as e:
        print(f"{run:<14} -> {type(e).__name__}: {e}")

In [ ]:
def snapshot_redshift(path):
    """Read the redshift from a SOAP file's header, trying the usual locations."""
    f = root[path]
    for grp in ("Header", "Cosmology", "SWIFT/Header", "SWIFT/Cosmology"):
        try:
            attrs = dict(f[grp].attrs)
        except Exception:
            continue
        for key in ("Redshift", "redshift", "Z", "z"):
            if key in attrs:
                return grp, key, float(np.atleast_1d(attrs[key])[0])
    return None, None, None

def soap(group, run, snap):
    return f"FLAMINGO/{group}/{run}/SOAP-HBT/halo_properties_{snap}.hdf5"

print("checking candidate snapshots:\n")
for group, run, snap in [("L1_m8", "L1_m8", "0078"),
                         ("L1_m9", "L1_m9", "0077"),
                         ("L1_m9", "L1_m9", "0076")]:
    p = soap(group, run, snap)
    try:
        grp, key, z = snapshot_redshift(p)
        print(f"{run:<8} {snap}  z = {z}   (from {grp}/{key})" if z is not None
              else f"{run:<8} {snap}  redshift attr not found; groups: {list(root[p])[:8]}")
    except Exception as e:
        print(f"{run:<8} {snap}  {type(e).__name__}: {e}")

In [ ]:
SNAP_M8 = "0078"
SNAP_M9 = "0077"     

RUNS = {
    "L1_m9":       soap("L1_m9", "L1_m9",       SNAP_M9),   # hydro_fiducial
    "Jet":         soap("L1_m9", "Jet",         SNAP_M9),   # hydro_jet_agn
    "fgas-4sigma": soap("L1_m9", "fgas-4sigma", SNAP_M9),   # hydro_stronger_agn
    "Planck":      soap("L1_m9", "Planck",      SNAP_M9),   # hydro_planck
}

soap_m9 = root[RUNS["L1_m9"]]
print("opened", RUNS["L1_m9"])

In [ ]:
flp.print_schema_report(soap_m9)

In [ ]:
n_rows = flp._shape_of(soap_m9, flp.FIELD_PATHS["stellar_mass"])[0]
print(f"catalogue rows: {n_rows:,}")

t0 = time.time()
fields_s = flp.load_flamingo_fields(soap_m9, selection=slice(0, 200_000))
dt = time.time() - t0
print(f"200k rows in {dt:.1f}s  ->  full load ~{dt*n_rows/200_000/60:.1f} min")

sample_s = flp.generate_flamingo_sample(fields_s, verbose=False)
pairs_s  = flp.find_flamingo_pairs(sample_s, r_min_kpc=500.0, r_max_kpc=1000.0, verbose=False)
print(f"smoke-test pairs from 200k rows: {len(pairs_s['i']):,}")

In [ ]:
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"
with open(cache_dir / "flamingo_sample.pkl", "rb") as f: s8 = pickle.load(f)
with open(cache_dir / "flamingo_pairs.pkl",  "rb") as f: p8 = pickle.load(f)
sh8 = p8["have_same_host"].astype(bool)
print("L1_m8 (run_001):")
print(f"  galaxies : {len(s8['stellar_mass']):,}")
print(f"  pairs    : {len(p8['i']):,}   same-host {sh8.sum():,}   diff-host {(~sh8).sum():,}")
print("  (142,713 survive dropna on all 11 features in the notebooks)")

In [ ]:
cfg = flp.load_selection_config(str(PROJECT / "analogues/flamingo_selection_config.json"))

t0 = time.time()
fields9, sample9, pairs9 = flp.run_flamingo_pipeline_from_config(cfg, soap_file=soap_m9)
sh9 = pairs9["have_same_host"].astype(bool)
print(f"\nL1_m9 fiducial — {(time.time()-t0)/60:.1f} min")
print(f"  galaxies : {len(sample9['stellar_mass']):,}")
print(f"  pairs    : {len(pairs9['i']):,}   same-host {sh9.sum():,}   diff-host {(~sh9).sum():,}")

In [ ]:
out = PROJECT / "analogues/cache/run_002_m9_fiducial"
out.mkdir(parents=True, exist_ok=True)
with open(out / "flamingo_sample.pkl", "wb") as f: pickle.dump(sample9, f)
with open(out / "flamingo_pairs.pkl",  "wb") as f: pickle.dump(pairs9, f)
with open(out / "provenance.txt", "w") as f:
    f.write(f"source: {RUNS['L1_m9']}\nconfig: flamingo_selection_config.json\n"
            f"pipeline: analogues/flamingo_lg_pipeline.py\n")

print(f"{'run':<14} {'galaxies':>10} {'pairs':>10} {'same':>9} {'diff':>9}")
print("-" * 56)
print(f"{'L1_m8':<14} {len(s8['stellar_mass']):>10,} {len(p8['i']):>10,} {sh8.sum():>9,} {(~sh8).sum():>9,}")
print(f"{'L1_m9':<14} {len(sample9['stellar_mass']):>10,} {len(pairs9['i']):>10,} {sh9.sum():>9,} {(~sh9).sum():>9,}")
print(f"\nsaved to {out}")

In [ ]:
cfg = flp.load_selection_config(str(PROJECT / "analogues/flamingo_selection_config.json"))

soap_jet = root[RUNS["Jet"]]
print("opened", RUNS["Jet"])

t0 = time.time()
fields_j, sample_j, pairs_j = flp.run_flamingo_pipeline_from_config(cfg, soap_file=soap_jet)
shj = pairs_j["have_same_host"].astype(bool)
print(f"\nL1_m9 Jet — {(time.time()-t0)/60:.1f} min")
print(f"  galaxies : {len(sample_j['stellar_mass']):,}")
print(f"  pairs    : {len(pairs_j['i']):,}   same-host {shj.sum():,}   diff-host {(~shj).sum():,}")

out_j = PROJECT / "analogues/cache/run_003_m9_jet"
out_j.mkdir(parents=True, exist_ok=True)
with open(out_j / "flamingo_sample.pkl", "wb") as f: pickle.dump(sample_j, f)
with open(out_j / "flamingo_pairs.pkl",  "wb") as f: pickle.dump(pairs_j, f)
with open(out_j / "provenance.txt", "w") as f:
    f.write(f"source: {RUNS['Jet']}\nconfig: flamingo_selection_config.json\n"
            f"pipeline: analogues/flamingo_lg_pipeline.py\n")
print(f"\nsaved to {out_j}")

In [ ]:
def load_run(name):
    d = PROJECT / "analogues/cache" / name
    with open(d / "flamingo_sample.pkl", "rb") as f: s = pickle.load(f)
    with open(d / "flamingo_pairs.pkl",  "rb") as f: p = pickle.load(f)
    return s, p

rows = [("L1_m8",      "run_001_mstar2e10"),
        ("L1_m9 fid",  "run_002_m9_fiducial"),
        ("L1_m9 Jet",  "run_003_m9_jet")]

print(f"{'run':<14} {'galaxies':>10} {'pairs':>10} {'same':>9} {'diff':>9} {'same %':>8}")
for label, d in rows:
    s, p = load_run(d)
    sh = p["have_same_host"].astype(bool)
    n  = len(p["i"])
    print(f"{label:<14} {len(s['stellar_mass']):>10,} {n:>10,} "
          f"{sh.sum():>9,} {(~sh).sum():>9,} {100*sh.sum()/n:>7.1f}%")

In [ ]:
def nstars_summary(sample, fields, label):
    """Report n_stars distribution for the final sample. Tries sample[...] first
    (exact final-sample galaxies); falls back to fields[...] filtered to the
    sample's stellar mass range if sample doesn't carry n_stars directly."""
    if "n_stars" in sample:
        n = np.asarray(sample["n_stars"])
        source = "sample (exact final-sample galaxies)"
    else:
        # fallback: filter raw fields to the sample's stellar mass range
        smin, smax = sample["stellar_mass"].min(), sample["stellar_mass"].max()
        mask = (fields["stellar_mass"] >= smin) & (fields["stellar_mass"] <= smax) \
               & (fields["n_stars"] >= 100)  # match the quality cut already applied
        n = np.asarray(fields["n_stars"])[mask]
        source = "fields, mass-window + quality filtered (approximate — not exact final-sample match)"

    print(f"{label}")
    print(f"  source: {source}")
    print(f"  N galaxies checked : {len(n):,}")
    print(f"  median n_stars     : {np.median(n):.0f}")
    print(f"  min n_stars        : {n.min():.0f}")
    print(f"  fraction < 150     : {(n < 150).mean():.1%}")
    print()

nstars_summary(sample9, fields9, "L1_m9 fiducial")
nstars_summary(sample_j, fields_j, "L1_m9 Jet")

print("L1_m8 (reference, from original resolution check):")
print("  median n_stars : 331")
print("  min n_stars    : 112")

In [ ]:
MASS_MIN, MASS_MAX = 2e10, 5e11  

def raw_vs_quality_attrition(fields, label):
    in_window_raw = (fields["stellar_mass"] >= MASS_MIN) & (fields["stellar_mass"] <= MASS_MAX)
    n_raw = in_window_raw.sum()

    in_window_quality = in_window_raw & (fields["n_stars"] >= 100)
    n_quality = in_window_quality.sum()

    removed = n_raw - n_quality
    print(f"{label}")
    print(f"  raw galaxies in mass window (any n_stars) : {n_raw:,}")
    print(f"  passing n_stars>=100 within that window    : {n_quality:,}")
    print(f"  removed for n_stars<100                     : {removed:,} ({removed/n_raw:.1%})")
    print()

raw_vs_quality_attrition(fields9, "L1_m9 fiducial")
raw_vs_quality_attrition(fields_j, "L1_m9 Jet")

In [ ]:
print("FLAMINGO top level:", sorted(list(root["FLAMINGO"])))
print()

for grp in sorted(list(root["FLAMINGO"])):
    try:
        inner = sorted(list(root[f"FLAMINGO/{grp}"]))
        print(f"FLAMINGO/{grp}  ({len(inner)} entries)")
        for n in inner:
            print("   ", n)
    except Exception as e:
        print(f"FLAMINGO/{grp} -> {type(e).__name__}: {e}")
    print()

# what we actually have per galaxy / per pair
try:
    print("sample keys:", sorted(sample9.keys()))
    print("pairs  keys:", sorted(pairs9.keys()))
except NameError:
    import pickle
    d = PROJECT / "analogues/cache/run_002_m9_fiducial"
    with open(d / "flamingo_sample.pkl", "rb") as f: sample9 = pickle.load(f)
    with open(d / "flamingo_pairs.pkl",  "rb") as f: pairs9  = pickle.load(f)
    print("(reloaded from cache)")
    print("sample keys:", sorted(sample9.keys()))
    print("pairs  keys:", sorted(pairs9.keys()))

In [ ]:
import numpy as np

def load_nstars(soap_file):
    return np.asarray(soap_file["BoundSubhalo/NumberOfStarParticles"][...])

def pair_arrays(sample, pairs, nstars_full):
    i, j = np.asarray(pairs["i"]), np.asarray(pairs["j"])
    ns   = nstars_full[np.asarray(sample["row_index"])]
    sd   = np.asarray(sample["vel_disp"])
    if sd.ndim > 1:                                  # dispersion matrix -> scalar
        sd = np.sqrt(sd[:, :3].sum(axis=1) / 3.0)
    return dict(n_min = np.minimum(ns[i], ns[j]),
                sigma = 0.5 * (sd[i] + sd[j]),
                theta = np.asarray(pairs["log_tot_virial_mass"]),
                ns_gal = ns)

def slope(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 50:
        return np.nan, int(m.sum())
    return float(np.polyfit(x[m], y[m], 1)[0]), int(m.sum())

runs = {}
for label, soap_f, smp, prs in [("fiducial", soap_m9,  sample9, pairs9),
                                ("Jet",      soap_jet, sample_j, pairs_j)]:
    t0 = time.time()
    ns_full = load_nstars(soap_f)
    runs[label] = pair_arrays(smp, prs, ns_full)
    print(f"{label:<9} n_stars read in {time.time()-t0:.0f}s   "
          f"sample median n_stars = {np.median(runs[label]['ns_gal']):.0f}   "
          f"(sanity: expect ~156 fid / ~150 Jet)")

print(f"\n{'n_stars >=':>11} {'fid pairs':>10} {'Jet pairs':>10} "
      f"{'fid slope':>11} {'Jet slope':>11} {'Jet-fid':>10}")
print("-" * 68)
for thr in [100, 150, 200, 300, 400, 500]:
    row = [f"{thr:>11}"]
    sl = {}
    for label in ("fiducial", "Jet"):
        d = runs[label]
        k = d["n_min"] >= thr
        sl[label], n = slope(d["sigma"][k], d["theta"][k])
        row.append(f"{n:>10,}")
    row.append(f"{sl['fiducial']:>11.5f}")
    row.append(f"{sl['Jet']:>11.5f}")
    row.append(f"{sl['Jet']-sl['fiducial']:>+10.5f}")
    print("".join(row))

In [ ]:
rng = np.random.default_rng(0)

def boot_slope(x, y, n_boot=1000):
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = len(x)
    out = np.empty(n_boot)
    for b in range(n_boot):
        k = rng.integers(0, n, n)
        out[b] = np.polyfit(x[k], y[k], 1)[0]
    return out

print("reference — FLAMINGO m8 vs TNG300: -0.00120, 95% CI [-0.00167, -0.00071]\n")
for thr in [100, 150, 200, 300]:
    bs = {}
    for label in ("fiducial", "Jet"):
        d = runs[label]
        k = d["n_min"] >= thr
        bs[label] = boot_slope(d["sigma"][k], d["theta"][k])
    diff   = bs["Jet"] - bs["fiducial"]
    lo, hi = np.percentile(diff, [2.5, 97.5])
    print(f"n_stars>={thr:<4} Jet-fid = {diff.mean():+.5f}  "
          f"95% CI [{lo:+.5f}, {hi:+.5f}]  "
          f"{'SIGNIFICANT' if lo*hi > 0 else 'not significant'}")